In [ ]:
import numpy as np
import os
from astropy.io import fits
from astropy.table import Table
import xarray as xr
import pandas as pd

In [ ]:
 



# ============================================================
# Helper function
# ============================================================

def load_fits_table(path):
    """
    Load a FITS binary table into a pandas DataFrame
    and replace dots in column names with underscores.
    """
    
    # Read FITS table
    table = Table(fits.open(path)[1].data)

    # Convert to pandas
    df = table.to_pandas()

    # Clean column names
    df.columns = [col.replace('.', '_') for col in df.columns]

    # Use galaxy ID as index
    df = df.set_index('id')

    return df


# ============================================================
# Load datasets
# ============================================================

obs = load_fits_table(
    '/project/galaxies/tjuchau/projects/CLOUDY_scripts/Galaxy_class_project3/run_AGN/out/observations.fits'
)

noagn = load_fits_table(
    '/project/galaxies/tjuchau/projects/CLOUDY_scripts/Galaxy_class_project3/run_noAGN/out/results.fits'
)

agn = load_fits_table(
    '/project/galaxies/tjuchau/projects/CLOUDY_scripts/Galaxy_class_project3/run_AGN/out/results.fits'
)


# ============================================================
# Convert to xarray datasets
# ============================================================

ds_obs = xr.Dataset.from_dataframe(obs).expand_dims(model=['obs'])

ds_noagn = xr.Dataset.from_dataframe(noagn).expand_dims(model=['noAGN'])

ds_agn = xr.Dataset.from_dataframe(agn).expand_dims(model=['AGN'])


# ============================================================
# Combine into one Dataset
# ============================================================

ds = xr.concat(
    [ds_obs, ds_noagn, ds_agn],
    dim='model'
)


# ============================================================
# Optional cleanup
# ============================================================

# Rename dimension from "id" -> "galaxy"
ds = ds.rename({'id': 'galaxy'})


In [ ]:
names = ['NGC5055', 'NGC4151', 'NGC1068', 'NGC0628', 'NGC1097']
ds.sel(galaxy=names[0])['best_reduced_chi_square']

In [ ]:
t = Table.read('/project/galaxies/tjuchau/projects/CLOUDY_scripts/Galaxy_class_project3/cigale_dustpedia_input_test.csv')


In [ ]:
t